### [1] Setup & Import

In [20]:
from contextlib import redirect_stdout
from datetime import datetime
from scipy.interpolate import pchip_interpolate
import plotly.express as px
import pandas as pd
import numpy as np
import duckdb
import base64
import json
import io
import re
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../config/SECRET.env') 
cust_salt = int(os.getenv('CUST_SALT_KEY'))
prod_salt = int(os.getenv('PRODUCT_SALT_KEY'))
ran_seed  = int(os.getenv('RANDOM_SEED'))

with open('../config/DYNAMIC_PRICE.json', 'r') as js:
    scale_dict = json.load(js)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

### [2] Profiling

In [21]:
def is_boolean(sample: pd.Series)-> bool:
    if sample.empty: return False
    if sample.dtype == 'bool': return True
    s = sample.astype('string').str.strip().str.lower()
    x = s.isin(['true','false','yes','no','1','0']).mean()
    y = s.nunique() 
    return (x >= 0.9) and (y <= 2)

def is_datetime(sample: pd.Series)-> bool:
    if sample.empty: return False
    if sample.dtype.kind in ['b','i','u','f']: return False
    if sample.dtype.kind == 'M': return True
    x = pd.to_datetime(sample, format='mixed', errors='coerce').notna().mean()
    y = sample.str.contains(r'[-/: ]', na=False).mean()
    return (x >= 0.9) and (y >= 0.9)

def is_alo(sample: pd.Series)-> bool:
    if sample.empty: return False
    s = sample.dropna().astype('string')
    x = s.str[0].isin(['0','3','5','7','8','9']).mean()
    y = s.str.replace(r'[,\-\s]', '', regex=True).str.len().mean()
    return (x >= 0.9) and (9 <= y <= 11)

def is_money(sample: pd.Series)-> bool:
    if sample.empty: return False
    if not sample.dtype.kind == 'o': return False
    pattern = r'^\s*-?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*$'
    return sample.astype('string').str.strip().str.match(pattern, na=False).mean() >= 0.9

def is_numeric(sample: pd.Series)-> bool:
    if sample.empty: return False
    return pd.to_numeric(sample, errors='coerce').notna().mean() >= 0.9

def is_category(sample: pd.Series)-> bool:
    if sample.empty: return False
    
    numeric_ratio = pd.to_numeric(sample, errors='coerce').notna().mean()
    nunique = sample.nunique()

    if numeric_ratio > 0.8:
        return False
    return 2 <= nunique <= 33

#!  stage_1: main
def stage_0(df: pd.DataFrame):
    #! stage_0: CAT_BY_NAME
    rules = {
    'date_col'    : r'(?:^|_)(?:date|ngay|day|created|updated)(?:_|$)',
    'time_col'    : r'(?:^|_)(?:time|gio|hour|minute|second|timestamp)(?:_|$)',
    'price'       : r'(?:^|_)(?:price|pice|unit_price|unitprice|đơn_giá|đơn_gia|gia|giá|gia_ban|giá_bán|prc)(?:_|$)',
    'numeric_col' : r'(?:^|_)(?:cost|qty|quantity|sl|disc|discount|percent|fee|rate|tax|shipping)(?:_|$)',
    'revenue'     : (r'(?:^|_)(?:revenue|total|total_amount|total_revenue|thanh_tien|thanhtien|'
                    r'doanh_thu|doanhthu|tổng_tiền|tong_tien|tongtien|grand_total|subtotal|tt|'
                    r'(?<!disc_)(?<!tax_)(?<!fee_)(?<!paid_)(?<!ship_)amount)(?:_|$)' ),
    'boolean_col' : r'(?:^|_)(?:ins_stt|tg|is_|status|active)(?:_|$)',
    'category_col': r'(?:^|_)(?:cat|type|category)(?:_|$)',
    'string_col'  : (r'(?:^|_)(?:ean|invoice|inv(?:oice|_no|_number)?|order_id|transaction_id|'
                    r'bill_no|bill_number|ma_hoa_don|serial|sku|upc|code|id)(?:_|$)' )
            }

    results = {key: [] for key in rules} | {key: [] for key in ['phone_col']}
    colname = df.columns.astype('string')

    #* all_true phát 1 vé True cho mỗi cột trong df.columns
    #*  - 
    # Each column only has one True ticket.
    all_true = pd.Series([True] * len(colname))
    
    for pocket, regex in rules.items():
        # Tạo mask: str.contains() trả True/False khi tên cột CHỨA item[1] rule
        boole = colname.str.strip().str.contains(regex, case=False, na=False, regex=True)
        # Update the True from boole mask with the False from all_true
        boole = boole & all_true
        # If not convert result to_list(), it wwould return like: 'date_col': Index(['date'], dtype='string')
        results[pocket] = colname[boole].to_list()
        # ~boole = match equal False
        all_true = all_true & ~boole
        
    #* Cái gì đi qua loop cũng chả còn vẹn nguyên :)
    # all_true down here is not all_true anymore
    pending_cols = colname[all_true].to_list()
    return df, results, pending_cols   
def stage_1(output_stage_0)-> dict:
    #! stage_1: CAT_BY_SAMPLE_DATA
    pocket_func = {
        'phone_col'   : [is_alo],
        'date_col'    : [is_datetime],
        'boolean_col' : [is_boolean],
        'numeric_col' : [is_money, is_numeric],
        'category_col': [is_category] 
    }
    #! unpack các biến từ stage_0
    df, results, pending_cols = output_stage_0

    # Chỉ xử lý các cột pending từ stage_0
    # flag ở đây là gì chả quan trọng vì flag reset mỗi loop
    for col in pending_cols:
        # series = get_sample(df[col])
        series = df[col].head(1000).dropna().head(500)
        flag = False
        for pocket, func in pocket_func.items():
            # print(f'{col} >>> tới "{pocket}"')
            for f in func:
                # print(f'{col} thử {f.__name__}')
                if f(series):
                    results[pocket].append(col)
                    # print(f"+ Nhập '{col}' vào: {pocket} vì khớp {f.__name__} flag = True")
                    flag = True
                    break
            if flag:
                break
            
        if not flag:
            results['string_col'].append(col)
            print(f'[Stage_1] no match col: {col} + -> string_col')
    return results


### [3] Validation

In [22]:
# --- Nhóm Revenue ---      
def cal_rev(df: pd.DataFrame, mask: pd.Series, payment_cols: list=None, mode: str='do_not_cal', results: dict=None)-> pd.Series:
    # def có 2 mode:
    #    - 'do_not_cal': chỉ mượn hàm để passing Dictionary results và Conditions count(qty_col) and count(price_col) == 1 
    #    - 'something_else': trả về revenue thay thế, ưu tiên sum(payments) , if not payments: price * qty
    #* Attribute trong pandas (ghi nhớ concept)
    #  - ghi kèm metadata theo pandas object, trả về 1 dict có thể bao gồm ( dict, list, bool,....)
    # Ghi vào: obj.attrs['tên_biến'] = giá_trị
    # Lấy ra: print(obj.attrs['tên_biến'])
    # Xóa sạch: obj.attrs.clear()

    # Nhận results từ tham số truyền vào để tránh gọi lại stage_1 gây lỗi unpack
    qty_rules = {'qty', 'quantity', 'sl', 'so_luong'}
    list_price = results.get('price', [])

    list_qty = df.columns[[any(word in qty_rules for word in col.split('_')) or col in qty_rules for col in df.columns]]
    price_n_qty_equal_1 = len(list_price) == 1 and len(list_qty) == 1

    # Khởi tạo biến rev_alter empty và same ở if -> elif -> return (1 flow cover all cases) Tránh lỗi UnboundLocalError.
    rev_alter = pd.Series(np.nan, index=df.index)
    # mode default là bỏ qua tính rev_alter
    if mode != 'do_not_cal':
        if payment_cols:
            rev_alter = df.loc[mask, payment_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
        elif price_n_qty_equal_1:
            rev_alter = df.loc[mask, list_price[0]] * df.loc[mask, list_qty[0]]
            
    # Ghi kèm hướng dẫn sd, phải để cuối cùng chứ cho lên .loc là mất hết
    rev_alter.attrs['results'] = results
    rev_alter.attrs['price_n_qty_equal_1'] = price_n_qty_equal_1
    return rev_alter

#!  rev_validate: main
def rev_validate(df, payment_cols=None, results: dict=None)-> pd.DataFrame:
    #* def kiểm tra số lượng & chất lượng của cột Revenue
    #* Nếu rev_col == 0: Tạo df[rev] = cal_rev()
    #* Nếu rev_col == 1: nếu data.notna() >= 99% -> Tính mean(rev) so sánh mean(payments) | mean(price*qty)
    #* nếu data.notna() < 99% -> fillna() = cal_rev()

    mask = pd.Series(True, index=df.index)
    cal_results = cal_rev(df, mask, payment_cols, mode='do_not_cal', results=results)
    price_n_qty_equal_1 = cal_results.attrs.get('price_n_qty_equal_1', False)

    key = 'revenue'
    list_rev = results.get(key, [])

    if len(list_rev) == 0:
        rev = key
        if rev not in df.columns:
            df[rev] = cal_rev(df, mask, payment_cols, mode='cal', results=results)
            print(f"[rev_validate] Created new column: {rev}")

    elif len(list_rev) == 1:
        rev = list_rev[0]
        rev_val = pd.to_numeric(df[rev], errors='coerce').notna().mean()
        rev_na = df[rev].isna().sum()
        print(f'[rev_validate] Detected: 1 revenue column ({rev_val*100:.2f}% valid)')
        
        # Tính toán giá trị thay thế cho toàn bộ bảng để dùng chung
        rev_alter_val = cal_rev(df, mask, payment_cols, mode='cal', results=results)
        
        if rev_val < 0.99:
            if payment_cols or price_n_qty_equal_1:
                mask = df[rev].isna()
                df.loc[mask, rev] = rev_alter_val.loc[mask]
                print(f'[rev_validate] Revenue gaps: {rev_na} filled')
            else:
                print(f'[rev_validate] Insufficient data to fill in {rev_na} revenue gaps.')

        # Tính mean để đối chiếu (reset mask về True để tính mean toàn cột)
        current_rev_numeric = pd.to_numeric(df[rev], errors='coerce')
        rev_mean = current_rev_numeric.mean()
        
        rev_alter_mean = rev_alter_val.mean()
        if (payment_cols or price_n_qty_equal_1) and rev_alter_mean != 0:
            print(f'[rev_validate] revenue(mean) / alter_revenue(mean) = {(rev_mean/rev_alter_mean)*100:.2f}%')
            
    else:
        print(f"[rev_validate] Cảnh báo: Có nhiều cột {key}: {list_rev}")
        
    return df

# ---- Nhóm Date -----
def chunks_maker(df: pd.DataFrame, ffilled_bfilled_date: str)-> dict:
#* Gom chunk bằng:  missing.indexs.diff().cumsum()
#* df = .zip(index, cumsum).groupby....> to_dict()
    # Indexs of NaT rows -> Index_Obj
    idx_of_nat = df.loc[df[ffilled_bfilled_date].isna(),:].index
    # Convert Index_Obj -> Series (values = Boolean after diff)
    series_error_diff = idx_of_nat.to_series().diff() > 1
    # Tổng tích lũy của Series Boolean (nếu index nối tiếp thì +0, đứt quãng +1)
    list_error_cumsum = series_error_diff.cumsum().to_list()
    # Tạo data_frame zip(index_lỗi, đánh dấu group lỗi)
    df_chunk = pd.DataFrame(zip(idx_of_nat, list_error_cumsum), columns=['idx', 'cumsum'])
    # Tạo dict groupby(cumsum)[idx] để tách các chunk ra
    df_chunk_to_dict = df_chunk.groupby('cumsum')['idx'].apply(list).to_dict()
    # Loại các chunk > 5 rows
    # filtered_chunks = {k: v if len(v) <= 3 else 'over the limit' for k, v in df_chunk_to_dict.items()}
    filtered_chunks = {k:v for k, v in df_chunk_to_dict.items() if len(v) <= 5}
    return filtered_chunks
def validate_n_correct_chunks(df: pd.DataFrame, dict_chunks: dict, raw_date: str, fill_date: str)-> list:
    pending_date_idx = []
    for _ , a_chunk in dict_chunks.items():
        p, n = min(a_chunk) - 1, max(a_chunk) + 1
        if p < 0 or n not in df.index:
            print(f"Skipping Chunk {a_chunk} = NA")
            continue
    #*------------------------
    # If prev or next is empty
        prev_pd, next_pd = df.at[p,fill_date], df.at[n,fill_date]
        if pd.isna(prev_pd) or pd.isna(next_pd):
            print(f'    prev or next is empty for chunk {a_chunk}')
            continue
    #*------------------------
    # Case 1: prev_pd == next_pd
        if prev_pd == next_pd:
            df.loc[a_chunk, fill_date] = prev_pd
            print(f'    [Match Both_day] Index {a_chunk} assigned {prev_pd.date()}')
            continue
    #*------------------------
    # Case 2.0: Stop if max gap < 4
        the_gap = (next_pd - prev_pd).days
        if the_gap >= 4:
            print(f'Chunk {a_chunk} has gap too large.')
            continue
    # Case 2.1: next_pd - prev_pd <= 3  and > 0
        if the_gap <= 3 and the_gap > 0:
    #*      Gom Set trước khi loop idx
            set_prev = {prev_pd.day, prev_pd.month, prev_pd.year}
            set_next = {next_pd.day, next_pd.month, next_pd.year}
            for idx in a_chunk:
                idx_raw = df.at[idx, raw_date]
                if pd.isna(idx_raw) or str(idx_raw).strip() == '':
                    pending_date_idx.append(idx)
                    print(f'    NGHI VẤN: Index {idx} nội dung là: {df.at[idx, raw_date]}')
                    continue
    #*      Similarity Check - (Set Validation)
                try:
                    idx_split =  idx_raw.replace(' ','/').replace('-','/').strip().split('/')
                    set_current = {int(x) for x in idx_split if x.strip().isdigit()}
                except(ValueError, TypeError, AttributeError):
                    pending_date_idx.append(idx)
                    continue

                not_prev, not_next = list(set_current - set_prev), list(set_current - set_next)

            # Nếu sau khi trừ set mà cả 2 dư ra 2 thành phần: continue
                if len(not_prev) >= 2 and len(not_next) >= 2:
                    pending_date_idx.append(idx)
                    print(f'    Something wrong at index: {idx} fill_date')
                    continue
            # If match Prev_date:
                if not not_prev:
                    df.at[idx, fill_date] = prev_pd
                    print(f'    [Match Prev_day] Index {idx} assigned {prev_pd.date()}')
            # If match Next_date:
                elif not not_next:
                    df.at[idx, fill_date] = next_pd
                    print(f'    [Match Next_day] Index {idx} assigned {next_pd.date()}')
                # Thay vì tự ghép, hãy đánh dấu để kiểm tra bằng tay
                else:
                    pending_date_idx.append(idx)
                    print(f'    NGHI VẤN: Index {idx} nội dung là: {df.at[idx, raw_date]}')
    return pending_date_idx

#!  recover_date: main
def recover_date(df: pd.DataFrame, date_raw: str, anchor_col_name: str=None)-> list:
    print(f'df.index: {df.index}')
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
        print("Warning: Index was not sorted. DataFrame has been sorted automatically.")

    list_if_error = []

    if date_raw:
        d1 = pd.to_datetime(df[date_raw], format='%Y-%m-%d', errors='coerce')
        d2 = pd.to_datetime(df[date_raw], format='%d-%m-%Y', errors='coerce')
        df['fill_date'] = d1.fillna(d2)

#* Nếu có Anchor_col > ffill, bfill trước.
    if anchor_col_name:
        df['fill_date'] = df.groupby(anchor_col_name)['fill_date'].transform('ffill')
        df['fill_date'] = df.groupby(anchor_col_name)['fill_date'].transform('bfill')
        print(f'    Missing date detected, proceed ffill & bffll by anchor "{anchor_col_name}"')

    number_of_errors_date = df['fill_date'].isna().sum()
    if number_of_errors_date > 0:   
        the_chunks = chunks_maker(df, 'fill_date')
        if the_chunks:
            list_if_error = validate_n_correct_chunks(df, the_chunks, date_raw, 'fill_date')
    if not list_if_error:
        print(f"    Number of NaT in 'fill_date': {df['fill_date'].isna().sum()}")
    return df['fill_date'], list_if_error

# ---- Nhóm Time -----
def time_format(df, time_col_name)-> pd.Series:      
    if time_col_name:
        t1 = pd.to_datetime(df[time_col_name], format='%I:%M%p' , errors='coerce')
        t2 = pd.to_datetime(df[time_col_name], format='%H:%M:%S', errors='coerce') 
        t3 = pd.to_datetime(df[time_col_name], format='%H:%M'   , errors='coerce') 

    return t1.fillna(t2).fillna(t3)


### [4] Execution

In [23]:
def execution(df: pd.DataFrame, final_results: dict)-> pd.DataFrame:
    df_new = pd.DataFrame()
    
    for key, cols in final_results.items():
        if not cols:
            continue
        # Loop through 
        if key == 'time_col':
            for c in cols:
                df_new[c] = time_format(df, c)
        # Vectorized on multiple columns  
        if key == 'date_col':
            df_new[cols] = df[cols]      
        if key == 'price':
            df_new[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
        if key == 'revenue':
            df_new[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
        if key == 'numeric_col':
            df_new[cols] = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0)
        if key == 'boolean_col':
            df_new[cols] = df[cols].astype('boolean')
        if key == 'category_col':
            df_new[cols] = df[cols].fillna('uncategorized').astype('category')
        if key in ['phone_col', 'string_col']:
            df_new[cols] = df[cols].astype('string').fillna('unknown')

    try:
        df_new = df_new[df.columns]
    except KeyError as e:
        print(f"🫠 Execution > Mất cột: {e}")
    return df_new

#### Customer_Master

In [24]:
def cus_normalize(df: pd.DataFrame, _p: str, _n: str, _e: str)-> pd.DataFrame:
    customer = [_p, _n, _e]
   
    #! 1. lower  >   replace(r'\s+')  >   strip
    df[customer] = df[customer].apply(lambda col: col.astype('string').str.lower().str.replace(r'\s+', ' ',regex=True).str.strip())

    #! 2. Filter email
    e_mask = df[_e].str.contains(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', regex=True)
    df[_e] = df[_e].where(e_mask, None)

    #! 3. phone.str.replace('\D'=not_a_number, '^84'=start_with: 84) 
    df[_p] = df[_p].str.replace(r'\D','', regex=True).str.replace(r'^84','0', regex=True)
    # Phone Masks
    p_len_equal_9      = df[_p].str.len() == 9
    p_len_less_9       = df[_p].str.len() < 9
    p_start_0          = df[_p].str.contains(r'^0', regex=True)
    p_start0_equal9    = p_len_equal_9 & p_start_0
    p_notstart0_equal9 = p_len_equal_9 & ~p_start_0
    p_lenmorethan_12      = df[_p].str.len() >= 12
    # Phone fixing
    df.loc[p_len_less_9, _p]       =      None                              # <9
    df.loc[p_start0_equal9, _p]    =      None                              # ^0 & ==9
    df.loc[p_notstart0_equal9, _p] = '0' + df.loc[p_notstart0_equal9, _p]   # ~^0 & ==9
    df.loc[p_lenmorethan_12, _p]   =      None

    #! Name.title()
    df[_n] = df[_n].replace('unknown', None).str.title()

    return df
def create_cust_master(df: pd.DataFrame, phone: str, name: str, email: str)-> pd.DataFrame:
    phone_name_email = [phone, name, email]
    return (df[phone_name_email].assign(
        name = lambda x: x[name].astype(str).str.title(),
        _p = lambda x: x[phone].astype(str).str.len(),
        _n = lambda x: x[name].astype(str).str.len(),
        _e = lambda x: x[email].astype(str).str.len() 
        ).sort_values(by=['_p','_n','_e'],
        axis= 0,
        ignore_index=True,
        ascending=[True,False,False]).groupby(phone, as_index=False)            
        .head(1)    
        .drop(['_p','_n','_e'], axis=1)
        .sort_values(by=[phone, name],
        ascending=[True, True])
        .reset_index(drop=True))            
    # Nếu groupby không có as_index=False thì nó sẽ cho phone=index
    # Nếu không có lệnh SELECT hoặc AGG() sau khi groupby, kết quả sẽ ở trạng thái super position (object quần què không hiển thị được ở dạng df)
    # Nếu sau groupby() có select [[col1,col2,.]] thì sẽ là 1 bản sao mới không tồn tại ['_p','_n','_e'] (không drop đc)
    # Groupby.head làm loạn index nên cuối cùng vẫn phải reset_index 
def base32_encode(df: pd.DataFrame, _p: str)-> pd.DataFrame:

    # 1. Encode phone
    df[_p] = pd.to_numeric(df[_p], errors='coerce').astype('Int64')
    df[_p] = (df[_p]
            .apply(lambda x: 
            base64.b32encode(
            int(x + cust_salt).to_bytes(5, 'big'))
            .decode().rstrip('=') if pd.notna(x) else None ))

    return df
def create_cus_id(df_cust_master: pd.DataFrame, _p: str, _n: str, _e: str)-> pd.DataFrame:
    df_cust_master.insert(0, 'id', 
    'CUS-' + (df_cust_master[_p].str[ :4] + '-' 
            + df_cust_master[_p].str[4: ])
            .str.upper())
    print(f'Created "id" column for df_cust_master')

    # Mask <Asterisk>@email
    df_cust_master[_e] = df_cust_master[_e].apply(lambda x: re.sub(r'(?<=^.)(.+)(?=.@)', lambda m: '*' * len(m.group(1)), str(x)))

    # Clean '<na>string'
    df_cust_master[_e] = df_cust_master[_e].str.replace('<na>', '', case=False)
    df_cust_master[_n] = df_cust_master[_n].str.replace('<na>', '', case=False).str.title()
    return df_cust_master

### [5] Control Center

In [25]:
# 1. Input
csv_2024     = r'../CSV_read_only/LMHN_2024_CLEAN.csv'
csv_2025     = r'../CSV_read_only/LMHN_2025_DIRTY.csv'
product_info = r'../CSV_read_only/DIGIBOX ALL PRICE - JAMES.csv'
config = {
    'payment_cols': ['cash', 'card', 'payoo', 'banking', 'mkt', 'vnpay', 'trade_in'],
    'date_anchor' : 'invoice',
    'date_pocket' : 'date',
    'drop_col'    : ['vat', 'note'],
    'anonymous'   : ['phone', 'name', 'email',]
    }


# 2. Load & Normalize
df_2024 = pd.read_csv(csv_2024)
df_2025 = pd.read_csv(csv_2025)
df_2024.columns = df_2025.columns

first_4_cols = df_2024.columns[:4]
df = pd.concat([df_2024, df_2025], axis=0, ignore_index=True).dropna(subset=first_4_cols, ignore_index=True)


df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
df = df.drop(config['drop_col'], axis=1, errors='ignore')


# 3. Cleaning 
f = io.StringIO()
with redirect_stdout(f):
    categorize_results = stage_1(stage_0(df))
    df = rev_validate(df, config['payment_cols'], categorize_results)
    df[config['date_pocket']], list_errors_date = recover_date(df, config['date_pocket'], config['date_anchor'])
    df = df.drop('fill_date', axis=1, errors='ignore')
    #* After cleaned
    df_new = execution(df, categorize_results)


# 4. Cust Master
cust_cols = config.get('anonymous')
_p, _n, _e = cust_cols

# Normalize before create Cust Master
df_new = cus_normalize(df_new, _p, _n, _e)
# Creating Cust Master
df_cust_master = create_cust_master(df_new, _p, _n, _e)
df_cust_master = base32_encode(df_cust_master, _p)
df_cust_master = create_cus_id(df_cust_master, _p, _n, _e)
# Encode main df[phone]
df_new = base32_encode(df_new, _p).drop([ _n, _e], axis=1, errors='ignore')

#* Map Cust Master back to main df
df_new = df_new.merge(df_cust_master, how='left', on=_p)
df_new.insert(1, 'month', df_new['date'].dt.month)
df_new.insert(2, 'year', df_new['date'].dt.year)

df_new

Created "id" column for df_cust_master


,date,month,year,invoice,sa,ean,cat,imei_sn,sap_article,sap_description,price,qty,ins_stt,ins_fee,disc_percent,disc_amount,revenue,cash,card,payoo,banking,mkt,vnpay,trade_in,phone,time,id,name,email
0,2024-04-01,4,2024,11176,Tom,4895229137837,3RD ACC,unknown,PHIDLP9859NW74,DISPLAY 15W WIRELESS POWER BANK 10000,1419000.0,1.0,False,0.0,0.0,0.0,1419000.0,0.0,1419000.0,0.0,0.0,0.0,0.0,0.0,None,1900-01-01 22:35:00,NaN,NaN,NaN
1,2024-04-01,4,2024,11177,Tom,195949121302,ACCESSORIES (APPLE),unknown,APPMUVV3ZA/A,20W USB-C POWER ADAPTER-ITS,549000.0,1.0,False,0.0,0.0,0.0,549000.0,0.0,0.0,0.0,0.0,0.0,549000.0,0.0,None,NaT,NaN,NaN,NaN
2,2024-04-01,4,2024,11178,Tom,194253686958,ACCESSORIES (APPLE),unknown,APPMQLU3ZP/A,USB-C TO APPLE PENCIL ADAPTER -ITP,349000.0,1.0,False,0.0,0.0,0.0,349000.0,349000.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaT,NaN,NaN,NaN
3,2024-04-01,4,2024,11179,Charlie,4895229104112,3RD ACC,unknown,PHIDLC5545V97,MFI SYNC CHARGE C TO LIGHTNING CAB 10M,490000.0,1.0,False,0.0,0.0,0.0,490000.0,0.0,0.0,0.0,0.0,0.0,490000.0,0.0,None,NaT,NaN,NaN,NaN
4,2024-04-01,4,2024,11180,Charlie,6945764522556,3RD ACC,unknown,MPW6945764522556,Miếng dán cường lực Mipow Kingbull HD Premium-...,295000.0,1.0,False,0.0,0.0,0.0,390000.0,0.0,390000.0,0.0,0.0,0.0,0.0,0.0,None,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17756,2025-12-31,12,2025,22361,Lee,195950664164.0,ACCESSORIES (APPLE),unknown,APPMGFW4FE/A,IPHONE 17 PRO MAX CL CASE-FAE,1403000.0,1.0,False,0.0,0.0,0.0,1403000.0,0.0,1403000.0,0.0,0.0,0.0,0.0,0.0,None,1900-01-01 21:18:00,NaN,NaN,NaN
17757,2025-12-31,12,2025,22361,Lee,6954661873388.0,3RD ACC,unknown,JCP6954661873388,Cường Lực JCPAL Chống Nhìn Trộm iPhone 17 - 17...,450000.0,1.0,False,0.0,0.0,0.0,450000.0,0.0,450000.0,0.0,0.0,0.0,0.0,0.0,None,1900-01-01 21:18:00,NaN,NaN,NaN
17758,2025-12-31,12,2025,22362,Lee,6945764527629.0,3RD ACC,unknown,MPW6945764527629,MagSafe MAT Case iP17 6.9 VN25 ORG,550000.0,1.0,False,0.0,0.0,0.0,550000.0,0.0,550000.0,0.0,0.0,0.0,0.0,0.0,None,1900-01-01 17:29:00,NaN,NaN,NaN
17759,2025-12-31,12,2025,22362,Lee,6945764527100.0,3RD ACC,unknown,MPW6945764527100,HD Anti-Spy Silk Premium iP17 6.9 VN25,450000.0,1.0,False,0.0,0.0,0.0,450000.0,0.0,450000.0,0.0,0.0,0.0,0.0,0.0,None,1900-01-01 17:48:00,NaN,NaN,NaN


### Traffic

In [26]:
store_traffic_2024 = duckdb.sql(
    "SELECT Period::DATE as Period, Traffic     FROM '../CSV_read_only/APPLE_2024_TRAFFIC.csv'")
store_traffic_2025 = duckdb.sql(
    "SELECT Period::DATE as Period, Traffic     FROM '../CSV_read_only/APPLE_2025_TRAFFIC.csv'")
store_traffic_2026 = duckdb.sql(
    "SELECT Period::DATE as Period, Traffic     FROM '../CSV_read_only/APPLE_2026_TRAFFIC.csv'")
holiday_vn = duckdb.sql(
    "SELECT Period::DATE as Period, Event_name  FROM '../CSV_read_only/Holiday_VN_2024_2026.csv'")

total_traffic = duckdb.sql(
"""--sql
WITH traffic AS(
    SELECT * FROM store_traffic_2024
        UNION ALL
    SELECT * FROM store_traffic_2025
        UNION ALL
    SELECT * FROM store_traffic_2026
)
SELECT 
    t.Period, 
    t.Traffic,
    h.Event_name
FROM  traffic t
LEFT JOIN holiday_vn h
ON t.Period = h.Period
ORDER BY t.Period ASC;
""").df()

traffic_scale = [1.12, 1.18]
def nonlinear_rescale(df:pd.DataFrame, metric:str, scale_min_max:list=None):
    if not is_numeric(df[metric]): return
    np.random.seed(ran_seed)
    min_raw = df[metric].nsmallest(2).values[1]
    max_raw = df[metric].nlargest(1).values[0]

    interp_index = np.interp(df[metric], [min_raw, max_raw], scale_min_max)
    noise = pd.Series(np.random.normal(1, 0.31, size=len(df)), index=df.index)
    final_index = interp_index  * np.sin((np.pi /2) ** 0.2) * noise
    processed = df[metric] * final_index

    if f'new_{metric}' in df.columns:
        df.drop(f'new_{metric}', axis=1, inplace=True)
    df.insert(1, f'new_{metric}', processed)

    # Fill low
    low_threshold:float = df[f'new_{metric}'].quantile(0.15)
    is_low = df[f'new_{metric}'] <= low_threshold
    df.loc[is_low, f'new_{metric}'] = np.nan

    df[f'new_{metric}'] = (df[f'new_{metric}']
        .interpolate(method='pchip')
        .round(-1)).astype('Int32')
    if metric in df.columns:
        df.drop(metric, axis=1, inplace=True)
    return df

total_traffic = nonlinear_rescale(total_traffic, 'Traffic', traffic_scale)





##### "All sensitive business data, including pricing structures and product identifiers, have been anonymized using statistical noise and cryptographic hashing. Customer data is shuffled to ensure compliance with NDA and data privacy standards."


In [27]:
product_master = pd.read_csv(product_info)
product_master.columns = product_master.columns.astype(
    'string').str.strip().str.replace(' ', '_').str.lower()


# drop blank EAN, drop duplicates EAN
product_master = product_master.dropna(
    subset='ean', axis=0).drop_duplicates(subset='ean', ignore_index=True)

# EAN To Numeric
product_master['ean'] = pd.to_numeric(
    product_master['ean'], errors='coerce')

check_ean = product_master['ean'].notna().mean()
if check_ean < 1:       #! Validation Warning
    print(f'EAN valid = {check_ean * 100}%')

if True:                #! Create 'master_sku'
    product_master.insert(1, 'master_sku',
                          product_master['ean'].apply(
                              lambda x: base64.b32encode(
                                  int(x + prod_salt).to_bytes(7, 'big')
                              ).decode().rstrip('=')
                              if not pd.isna(x) else None)
                          )

is_apple = product_master['cat'].isin(
    ['ACCESSORIES (APPLE)', 'IPAD', 'IPHONE', 'MAC', 'WATCH'])
if True:                #! Split 'master_sku' into 'APP' & '3RD'
    app_ean = 'APP-' + \
        product_master['master_sku'].str[2:7] + \
        '-' + product_master['master_sku'].str[7:]
    third_ean = '3RD-' + \
        product_master['master_sku'].str[2:7] + \
        '-' + product_master['master_sku'].str[7:]
    product_master['master_sku'] = np.where(is_apple, app_ean, third_ean)

if True:                #! Extract Color column
    color_keywords = [
        "black titanium", "white titanium", "desert titanium", "natural titanium",
        "black", "white", "silver", "gold", "space grey", "space gray", "midnight",
        "starlight", "pink", "blue", "green", "purple", "red", "yellow", "orange",
        "teal", "ultramarine", "clay", "guava", "cypress", "winter blue", "storm blue",
        "elderberry", "slate blue", "abyss blue", "dark cherry", "forest green", "ink",
        "umber", "lilac", "succulent", "sunglow", "olive", "soft mint", "light blue",
        "sunshine", "taupe", "mulberry", "pacific blue", "evergreen", "indigo",
        "pride edition", "bright orange", "clover", "moss green", "golden brown",
        "sequoia green", "wisteria", "marigold", "pink pomelo", "blue jay",
        "lemon zest", "eucalyptus", "nectarine", "blue fog", "english lavender",
        "marine blue", "canary yellow", "sky", "gray", "grey", "clear", "transparent",
        # viết tắt thường gặp ở 3RD ACC
        "blk", "gry", "pnk", "wht", "blu", "grn", "org", "ylw", "crm"
    ]

    def extract_color(row):
        text = str(row['product_name']) + " " + \
            str(row.get('sap_description', ''))
        text = text.lower().replace('/', ' ').replace('-', ' ')

        # Ưu tiên cụm dài trước
        for color in sorted(color_keywords, key=len, reverse=True):
            if color in text:
                # Map viết tắt về tên đẹp
                if color in ['blk', 'black']:
                    return 'Black'
                if color in ['gry', 'gray', 'grey']:
                    return 'Gray'
                if color in ['pnk']:
                    return 'Pink'
                if color in ['wht']:
                    return 'White'
                if color in ['blu']:
                    return 'Blue'
                if color in ['grn']:
                    return 'Green'
                if color in ['org']:
                    return 'Orange'
                if color in ['ylw']:
                    return 'Yellow'
                if color in ['crm']:
                    return 'Cream'
                return color.title().replace('gray', 'Gray').replace('grey', 'Gray')
        return None
    product_master['color'] = product_master.apply(extract_color, axis=1)
is_color = product_master['color'].notnull()
if True:                #! Extract Capacity & Size column
    regex = r'(?i)(\d{1,4})\s*(GB|TB|mm)'
    product_master['memory_size'] = (
        product_master['product_name'].str.extract(regex, expand=False)[0].fillna('') +
        product_master['product_name'].str.extract(
            regex, expand=False)[1].fillna('')
    )
    # Loại bỏ cho 3RD ACC và DEMO
    product_master.loc[product_master['cat'].str.contains(
        '3RD ACC|DEMO', case=False, na=False), 'memory_size'] = None
    # Dọn dẹp
    product_master['memory_size'] = product_master['memory_size'].replace([
                                                                          '', 'nan'], None)
    product_master['memory_size'] = product_master['memory_size'].str.replace(
        r'(\d+)([A-Za-z]+)', r'\1\2', regex=True)
is_memory = product_master['memory_size'].notnull()

is_apple_device = is_apple & is_color
if True:                #! Create 'new_product_name' and replace original
    product_master['new_product_name'] = np.where(is_apple_device,
                                                  product_master['detail_sub_lob']
                                                  + ' ' +
                                                  product_master['color']
                                                  + ' ' + np.where(is_memory, product_master['memory_size'], ''), None)

    is_sub_cat = product_master['detail_sub_lob'].notnull()
    non_apple = ~is_apple_device & is_color & is_sub_cat

    product_master['new_product_name'] = np.where(non_apple,
                                                  product_master['detail_sub_lob']
                                                  + ' ' + product_master['color'], product_master['new_product_name'])

    other = product_master['new_product_name'].isnull()
    product_master.loc[other,
                       'new_product_name'] = product_master['product_name']

    product_master['new_product_name'] = product_master['new_product_name'].str.strip(
    ).str.upper()
product_master['product_name'] = product_master['new_product_name']
#                       #! Create 'new_price' by static noise & scaling
def price_scale(df, cat_col: str, price_col: str, scale_dict: dict)-> pd.DataFrame:
    np.random.seed(ran_seed)
    for key, inner_dict in scale_dict.items():
        rate_min = inner_dict['scale'][0]
        rate_max = inner_dict['scale'][1]
        mag = inner_dict['mag']
        sfx = inner_dict['sfx']

        mask = df[cat_col] == key
        raw_coord = df.loc[mask, price_col]
        # Pick Second min() and First max()
        price_min = raw_coord.nsmallest(2).values[-1]
        price_max = raw_coord.nlargest(1).values[0]

        #* Create new_price | use np.floor to round down
        new_col = 'new_price' 
        noise = pd.Series(np.random.normal(1, 0.0112, size=len(raw_coord)), index=raw_coord.index)
        interp_factor = np.interp(
            raw_coord, 
            [price_min, price_max], 
            [rate_min, rate_max])
        
        new_price  = raw_coord * interp_factor * noise
        df.loc[mask, new_col] = np.floor(new_price / mag) * mag + sfx

    return df   

product_master = price_scale(product_master, 'cat', 'price', scale_dict)

#                       #! DROP unnecessary cols
final_drop = ['sap_article', 'sap_description', 'new_product_name', 'price']
product_master = product_master.drop(final_drop, axis=1, errors='ignore')
#! Save .CSV
product_master.to_csv('../CSV_write/Anonym_Price.csv', index=False, encoding='utf-8-sig')
print('saved ../CSV_write/Anonym_Price.csv')

saved ../CSV_write/Anonym_Price.csv


#### EDA [df_new]

In [28]:
# ! {**dict, ...} -> Unpacking for Merging > mở dict cho vào trong dict
# ! (**dict) -> Keyword Argument Unpacking > biến dict thành name_1=(parameter), name_2=....
#  .style xong thì chỉ để làm cảnh thôi, class=Styler
vnd = '{:,.0f} VNĐ'
qty = '{:,.0f}'

cat_config = {
    'total_revenue'  : ('revenue', 'sum'),
    'mean_revenue'   : ('revenue', 'mean'),
    'total_quantity' : ('qty', 'sum'),
    'cust_count'     : ('id', 'nunique')
}
format_dict = {
    'total_revenue': qty,
    'mean_revenue': qty,
    'total_quantity': qty,
    'cust_count': qty + ' khách'
}
cat_group = df_new.groupby('cat', observed=False).agg(**cat_config).reset_index(drop=False)
cat_group = cat_group.sort_values(by='total_revenue', axis=0, ascending=False, ignore_index=True)

groupby_config = {
    'total_revenue'  : ('revenue', 'sum'),
    'mean_revenue'   : ('revenue', 'mean'),
    'total_quantity' : ('qty', 'sum'),
    'cust_count'     : ('id', 'nunique')
}
by_month_cat = df_new.groupby(['month', 'cat'], observed=False).agg(**cat_config).reset_index(drop=False)
by_month_cat = by_month_cat.sort_values(by=['month', 'cat'], axis=0, ascending=[True, False], ignore_index=True)



#### SQL

In [ ]:
# ------ config ------
source_df         = 'df_new'
date_col          = 'date'
time_col          = 'time'
qty_col           = 'qty'
rev_column        = 'revenue'        
event_start_date  = '2025-09-18'    
analysis_end_date = '2025-12-31' 
rolling_window    = 6               
# ----------------------

# 1: Chuẩn hóa dữ liệu gốc về tên cột cố định (qty, date, time)
duck_new = duckdb.sql(f"""--sql
    SELECT *
    REPLACE (
        {qty_col}::INT32 as qty,
        {date_col}::DATE as date,
        {time_col}::TIMESTAMP as time
    )
    FROM {source_df};
""")
# Phải groupby cho mỗi ngày thành 1 dòng trước 
# vì AVG() nó không quan tâm 1 ngày có bao nhiêu dòng, 
# nếu 1 ngày càng nhiều dòng thì số AVG càng nhỏ và chả phản ánh đúng cái gì cả

duck_date_rev = duckdb.sql(f"""--sql
    WITH pre_cal AS (
        SELECT
            date as Date,
            SUM({rev_column}) as Revenue_by_Date                                            -- #!!!
        FROM duck_new
        GROUP BY date
        ORDER BY date ASC
    ),

    raw_cal AS (
        SELECT
            Date,
            Revenue_by_Date,
            ROUND(AVG(Revenue_by_Date) OVER(
                ORDER BY Date ROWS BETWEEN {rolling_window} PRECEDING AND CURRENT ROW
            ), 0) as Rolling_7Day            
        FROM pre_cal
    ),

    sqrt_abc AS (
        SELECT
            *,
            ROUND(SQRT(Revenue_by_Date), 0) as sqrt_revenue, 
            ROUND(SQRT(Rolling_7Day), 0)    as sqrt_rolling,
            CASE WHEN Date >= '{event_start_date}' AND Date <= '{analysis_end_date}' 
                 THEN ROUND(SQRT(Rolling_7Day), 0) ELSE NULL END as sqrt_event_launching,
            CASE WHEN Date >= '{event_start_date}' AND Date <= '{analysis_end_date}' 
                 THEN 1 ELSE 0 END as is_after_event
        FROM raw_cal
    ),

    mean_abc AS (
        SELECT 
            *,  
            AVG(CASE WHEN is_after_event = 0 THEN Revenue_by_Date ELSE NULL END) OVER() as rbm,
            AVG(CASE WHEN is_after_event = 1 THEN Revenue_by_Date ELSE NULL END) OVER() as ram
        FROM sqrt_abc
    )

    SELECT 
        *,
        CASE WHEN is_after_event = 0 THEN rbm ELSE NULL END as raw_before_event_mean,
        CASE WHEN is_after_event = 1 THEN ram ELSE NULL END as raw_after_event_mean,
        CASE WHEN is_after_event = 0 THEN SQRT(rbm) ELSE NULL END as sqrt_before_event_mean,
        CASE WHEN is_after_event = 1 THEN SQRT(ram) ELSE NULL END as sqrt_after_event_mean,
        (ram / rbm - 1) * 100 AS event_increase
    FROM mean_abc;
""").df()

if True: 
    number_cols = duck_date_rev.select_dtypes(include='number').columns
    duck_date_rev[number_cols] = duck_date_rev[number_cols].round(2).convert_dtypes()

duck_date_rev

### First Chart
Có thể add vào bất cứ chart nào\
fig.add_scatter( )\
fig.add_bar( )\
fig.add_histogram( )\
fig.add_vline( )\
fig.add_hline( )\
fig.add_annotation( )\
fig.update_traces( )\
fig.update_xaxes( )\
fig.update_yaxes( )\
fig.update_layout( )

In [30]:
tick_vals_raw = [
    0, 200_000_000, 600_000_000, 
    1_200_000_000,  1_800_000_000
    ]
y_max_sqrt = (max(tick_vals_raw) + 4e8) **0.5
tick_text_raw = ['0', '200 Tr', '600 Tr', '1,2 Tỉ', '1,8 Tỉ']
tick_vals_sqrt = np.sqrt(tick_vals_raw).tolist()

if True: #! Params
    #todo Không thể gán value cho vế dùng function/method
    x_raw               = 'Date'
    y_sqrt              = ['sqrt_revenue', 'sqrt_rolling']
    y_raw               = ['Revenue_by_Date', 'Rolling_7Day']
    legend_a            = 'Daily Revenue'
    legend_b            = '7D Trend'

    # title & range
    x_title           = 'Tháng'
    y_title           = 'VNĐ'
    x_range           = ["2025-01-01", "2025-12-31"]
    y_range           = [0, y_max_sqrt]

    # Event & Mean columns
    sqrt_event_end_year = 'sqrt_event_launching' 
    raw_b_mean          = 'raw_before_event_mean'
    raw_a_mean          = 'raw_after_event_mean'
    sqrt_b_mean         = 'sqrt_before_event_mean'
    sqrt_a_mean         = 'sqrt_after_event_mean'
    event_increase      = duck_date_rev.loc[0, 'event_increase']


fig = px.line(duck_date_rev, 
    title = '<b>AAR Store: Doanh thu 2025</b> <span style="font-size:14px; color:#888;">(VNĐ - sqrt scale)</span>',
    x = x_raw, 
    y = y_sqrt,                                       # Truyền list tên cột vào đây (cột để vẽ, có thể đã Square hoặc log)
    custom_data = y_raw, # Custom_data = Values Raw dùng để show kết quả hover. 
    labels = {'variable': '<b>Metric</b>'},            # Để đặt tên trục/ legend
    hover_name = x_raw,
    # color = None,
    color='variable',
    animation_frame=None, #! Test
    render_mode = 'svg'
    # width=1400, height=550
)

# Rolling 7D
fig.update_traces(
    selector={'name': y_sqrt[1]},
    name = legend_b,
    showlegend = True,
    line_color = "#4281AF", 
    line = dict(
        dash = 'solid',
        width = 2.8,
        shape = 'spline',
        smoothing = 1.3
        ),
    opacity = 1,
    fill = 'tozeroy', # Đổ màu từ đường line xuống trục 0
    fillcolor = 'rgba(120, 197, 239, 0.1)', 
    hovertemplate = "<b>Tuần: %{x|%W}</b><br>Trung bình: %{customdata[1]:,.0f} VNĐ<extra></extra>",
    # visible = 'legendonly'
)
# Daily Revenue
fig.update_traces(
    selector={'name': y_sqrt[0]},
    name = legend_a,
    showlegend = True,
    # line_color="#5EAEFF", 
    line_color = "#367050", 
    line = dict(
        width = 0.8, 
        shape = 'hv', 
        smoothing = 1.3
        ),
    opacity = 0.4,
    hovertemplate = "<b>Ngày: %{x|%d-%m-%Y}</b><br>Doanh thu: %{customdata[0]:,.0f} VNĐ<extra></extra>"
    # fill='tozeroy', # Đổ màu từ đường line xuống trục 0
    # fillcolor='rgba(135, 206, 250, 0.18)', # Màu xanh nhạt với độ trong suốt 20%
)
# Event_launch -> End year
fig.add_scatter(
    line_color = "#4281AF",
    x = duck_date_rev[x_raw], 
    y = duck_date_rev[sqrt_event_end_year],
    fill = 'tozeroy',
    # fillcolor='rgba(255, 167, 0, 0.2)',
    fillcolor = 'rgba(120, 200, 40, 0.2)',
    line = dict(width = 2.7,       # Ẩn = 0
                shape = 'spline', 
                smoothing = 1.3),    
    name = '',                      #! Ẩn
    showlegend = False,             # Giấu khỏi bảng chú thích (Legend)
    hoverinfo = 'skip'              # Rê chuột vào không hiện gì cả
)
# Mean line before_npi
fig.add_scatter(
    x=duck_date_rev[x_raw], 
    y=duck_date_rev[sqrt_b_mean], 
    mode='lines', 
    line=dict(color="#467797", width=1.7, dash='longdashdot'), # Chỉnh trực tiếp ở đây là chắc ăn nhất
        # BỘ BA "VÔ HÌNH":
    showlegend=False,        # 1. Giấu khỏi bảng chú thích (Legend)
    hoverinfo='skip',        # 2. Rê chuột vào không hiện gì cả
    name=''                  # 3. Để tên trống để tránh hiện lỗi nếu lỡ hover trúng
)
# Mean line after_npi
fig.add_scatter(
    x=duck_date_rev[x_raw], 
    y=duck_date_rev[sqrt_a_mean], 
    mode='lines', 
    line=dict(color="#4EA073", width=1.7, dash='longdashdot'), # Chỉnh trực tiếp ở đây là chắc ăn nhất
        # BỘ BA "VÔ HÌNH":
    showlegend=False,        # 1. Giấu khỏi bảng chú thích (Legend)
    hoverinfo='skip',        # 2. Rê chuột vào không hiện gì cả
    name=''                  # 3. Để tên trống để tránh hiện lỗi nếu lỡ hover trúng
)


fig.update_yaxes(
    tickmode = "array",
    tickvals = tick_vals_sqrt, # Vị trí đặt vạch chia (trên thang đo sqrt)
    ticktext = tick_text_raw, # Chữ hiển thị tại vạch đó
    tickprefix = None,
    ticksuffix = '    ',
    automargin = True,
    title = y_title,
    range = y_range,
)
fig.update_xaxes(
    ticklabelmode = "period",
    title_text = x_title,
    range = x_range, 
    title_font = dict(size=12),
    dtick = "M1",      # Có thể là 3 tháng, 1 ngày, 1 giờ...
    tickformat = "%b %y", # %b = chuẩn ISO cho tháng dạng Jan,...
    showgrid = True,
    gridwidth = 1, 
    gridcolor = 'white',
    # x_line = off
    showline = False, 
)
# Thêm đường Trần (y = max(y))
fig.add_hline(y = y_max_sqrt, 
              line_dash="solid", 
              line_color="#D3D3D3", 
              line_width=1)
# Thêm đường Sàn (ví dụ 0 VNĐ)
fig.add_hline(y=0, 
              line_dash="solid", 
              line_color="#D3D3D3", 
              line_width=0.5)
# Thêm Event Line
fig.add_vline(
    x = event_start_date, 
    line_width = 3.1, 
    line_dash = "longdash", 
    line_color = "#E8732A",
    layer = 'above'
)
fig.add_annotation(
    x = event_start_date, 
    y = 1.015, yref = "paper",
    text = "<b>🚀 iPhone 17 NPI</b><br>September 19, 2025.",
    align = 'left',
    showarrow = False,
    font = dict(
        family="Segoe UI Semibold",
        size=10,         
        color="#F28136",   
    ),
    xshift = -5,            # Đẩy chữ sang phải 5px để không dính sát vào đường kẻ
    yanchor = "bottom",
    xanchor = "left"
)
fig.add_annotation(
    text=f"<b>▲ {event_increase:.1f}%</b><br>Vượt mức trung bình trước NPI<br>duy trì đến hết năm 2025.",
    x='2025-10-21',  #! x theo df
    y=0.72,          #! y theo paper
    yref="paper",    # Bắt buộc có dòng này để y= có ý nghĩa
    showarrow=False,
    align="left",
    xanchor="left", 
    yanchor="top",  
    font=dict(size=10, color="gray")          
)

#! Theme | Legend | Margin | X-Y show title | Grid_color | Font | Hover_mode.
fig.update_layout(
    template="plotly_white",
    #! Legend
    legend=dict(
        orientation="h",        # Legend nằm ngang
        yanchor="top", y=1.175, 
        xanchor="center", x=0.5,
        font=dict(size=12, color="#86868b")
        ),
    #! Lề
    margin=dict(
        b=40,
        t=80, 
        l=40,  
        r=40),

    #! Font
    font_family='Segoe UI Semibold',
    font_color="#2C528C",
    title_font_size=26,

    #! X-axis
    xaxis=dict(
        title=None,             # Bỏ
        gridcolor="#D3D3D3",
        gridwidth = 0.2,
        showgrid=False,
        tickfont=dict(size=12),
        title_font=dict(size=12)
    ),
    #! Y-axis
    yaxis=dict(
        gridcolor = "#ECECEC",
        gridwidth = 0.1,
        showgrid = False,
        tickfont = dict(size=12),
        title_font = dict(size=12)
    ),
    hovermode="x unified",
    transition_duration=800,
    hoverlabel=dict(bgcolor="rgba(255, 255, 255, 0.9)", font_size=13),

# Thêm Button vào trong fig.update_layout

)


fig.show()
print((duck_date_rev['Date'].value_counts() > 1).sum())

0


Log

In [ ]:
# print(f.getvalue())
# now = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
# with open('../log/cleaning_log.txt', 'a', encoding='utf-8') as logg:
#     logg.write(f"\n--- Run at: {now} ---\n")
#     logg.write(f.getvalue())

# # print(f"[{'Stage_1':<15}] no match col: {'col':<20} -> string_col")